In [ ]:
"""in this day's learning ,i do think you should focus on the monte colro simualation and the black
shores method.

all this above is caring the sensitivity analysis. care about the parmameters and the local sensitivity,global sensitivity,
and so on


well,it is quite important to learn about the matplotlib to visualize your data analysis
of course, as a finance student, i should just read and learn about the financial part of matplotlib


回测过度拟合的信号（这个窗口拟合得很好，但是相邻窗口却拟合得很差，input的数据稍微变化一点，就产生剧烈波动，

敏感性分析：对使用的method做循环，观测绩效变化）

well，金融的关键可视化三件套
净值曲线（equity curve），
回撤图（drawdown chart），
年度收益柱状图：主要来看稳定性

在用plot画图的时候，两个参数，默认前边的参数画x值，后面的参数画y值,
matplotlib里的‘b-’表示的是画蓝色实线，‘b--’表示的画蓝色虚线
"""

In [ ]:
 def plot_equity_curve(self, benchmark: bool = True, figsize: tuple = (12, 6)):
        """绘制净值曲线（连续复利版本）"""
        perf = self.calculate_performance()
        cum_returns = perf['cum_returns']
        
        fig, ax = plt.subplots(figsize=figsize)
        ax.plot(cum_returns.index, cum_returns.values, 'b-', linewidth=1.5, label='Strategy')
        
        if benchmark:
            # 买入持有（连续复利）
            bh_log_returns = self.returns.mean(axis=1)
            cum_bh = bh_log_returns.cumsum().apply(np.exp)
            ax.plot(cum_bh.index, cum_bh.values, 'gray', linewidth=1, alpha=0.7, label='Buy & Hold')
        
        ax.axhline(y=1, color='black', linestyle='--', alpha=0.5)
        ax.legend()
        ax.set_title('Strategy Equity Curve')
        ax.set_ylabel('Net Value')
        ax.grid(True, alpha=0.3)
        
        return fig
    
    
    def plot_drawdown(self, figsize: tuple = (12, 4)):
        """绘制回撤图"""
        perf = self.calculate_performance()
        cum_returns = perf['cum_returns']
        cummax = cum_returns.cummax()
        drawdowns = (cum_returns - cummax) / cummax * 100
        
        fig, ax = plt.subplots(figsize=figsize)
        ax.fill_between(drawdowns.index, 0, drawdowns.values, color='red', alpha=0.5)
        #well,fillbetween（）x轴数据，y轴上界，y轴下界
        ax.set_title('Strategy Drawdown')
        ax.set_ylabel('Drawdown (%)')
        ax.grid(True, alpha=0.3)
        
        return fig
    
    
    def plot_monthly_heatmap(self, figsize: tuple = (12, 5)):
        """绘制月度收益热力图（连续复利版本）"""
        if self.strategy_returns is None:
            raise ValueError("请先计算策略收益")
        
        # 组合对数收益
        portfolio_log_returns = self.strategy_returns.sum(axis=1)
        
        # 对数收益按月求和
        monthly_log_returns = portfolio_log_returns.resample('ME').sum()
        
        # 转回简单收益率用于热力图展示
        monthly_returns = monthly_log_returns.apply(lambda x: np.exp(x) - 1)
        monthly_matrix = monthly_returns.groupby(
            [monthly_returns.index.year, monthly_returns.index.month]
        ).first().unstack()
        #这里将groupby依旧是，这里是双层行索引，
        #.first()表示取出这唯一的收益率
        #.unstack() 表示将一维序列转换为二维的DataFrame
        
        if monthly_matrix.empty:
            return None
        
        fig, ax = plt.subplots(figsize=figsize)
        im = ax.imshow(monthly_matrix.values, cmap='RdYlGn', aspect='auto',
                       vmin=-0.1, vmax=0.1)
        #imshow是专门用来画矩阵，热力图和图像的函数，cmap = ‘RDYlGn’则是专门有标准的颜色来表示热力图盈亏
        #aspect = 'auto'表示自动调整热力图的比例
        
        # 设置标签
        months = ['Jan','Feb','Mar','Apr','May','Jun',
                  'Jul','Aug','Sep','Oct','Nov','Dec']
        ax.set_xticks(range(len(monthly_matrix.columns)))
        ax.set_xticklabels(months[:len(monthly_matrix.columns)])
        ax.set_yticks(range(len(monthly_matrix.index)))
        ax.set_yticklabels(monthly_matrix.index)
        ax.set_title('Monthly Returns Heatmap')
        
        plt.colorbar(im, ax=ax, format=mtick.FuncFormatter(lambda x, _: f'{x:.0%}'))
        # .colorbar() 表示颜色具体表示多少数值，因为之前的热力图只有颜色来表示盈亏，却还不知道具体盈亏多少
        return fig
    
    
    def plot_full_report(self, title: str = 'Strategy Backtest Report',
                         figsize: tuple = (14, 12)):
        """生成完整回测报告（三合一）"""
        perf = self.calculate_performance()
        cum_returns = perf['cum_returns']
        portfolio_log_returns = self.strategy_returns.sum(axis=1)
        
        fig, axes = plt.subplots(3, 1, figsize=figsize)
        
        # 1. 净值曲线
        axes[0].plot(cum_returns.index, cum_returns.values, 'b-', linewidth=1.5)
        axes[0].fill_between(cum_returns.index, 1, cum_returns.values,
                              where=(cum_returns.values >= 1), color='green', alpha=0.3)
        axes[0].fill_between(cum_returns.index, 1, cum_returns.values,
                              where=(cum_returns.values < 1), color='red', alpha=0.3)
        axes[0].axhline(y=1, color='black', linestyle='--', alpha=0.5)
        axes[0].set_title(f'{title} - Equity Curve')
        axes[0].set_ylabel('Net Value')
        axes[0].grid(True, alpha=0.3)
        
        # 2. 回撤图
        cummax = cum_returns.cummax()
        drawdowns = (cum_returns - cummax) / cummax * 100
        axes[1].fill_between(drawdowns.index, 0, drawdowns.values, color='red', alpha=0.5)
        axes[1].set_title('Drawdown (%)')
        axes[1].set_ylabel('Drawdown %')
        axes[1].grid(True, alpha=0.3)
        
        # 3. 月度收益热力图
        monthly_log_returns = portfolio_log_returns.resample('ME').sum()
        monthly_returns = monthly_log_returns.apply(lambda x: np.exp(x) - 1)
        monthly_matrix = monthly_returns.groupby(
            [monthly_returns.index.year, monthly_returns.index.month]
        ).first().unstack()
        
        if not monthly_matrix.empty:
            im = axes[2].imshow(monthly_matrix.values, cmap='RdYlGn', aspect='auto',
                               vmin=-0.1, vmax=0.1)
            months = ['Jan','Feb','Mar','Apr','May','Jun',
                      'Jul','Aug','Sep','Oct','Nov','Dec']
            axes[2].set_xticks(range(len(monthly_matrix.columns)))
            axes[2].set_xticklabels(months[:len(monthly_matrix.columns)])
            axes[2].set_yticks(range(len(monthly_matrix.index)))
            axes[2].set_yticklabels(monthly_matrix.index)
            axes[2].set_title('Monthly Returns Heatmap')
            plt.colorbar(im, ax=axes[2], format=mtick.FuncFormatter(lambda x, _: f'{x:.0%}'))
        
        plt.tight_layout()
        return fig
    
    
    def parameter_sensitivity(self, param_name: str, param_range: list,
                              factor_func: Callable, threshold: float = 0,
                              **kwargs) -> pd.DataFrame:
        """
        参数敏感性分析（EWMA 友好版本）
        
        Parameters:
        param_name: 参数名称（如 'span', 'window'）
        param_range: 参数取值范围
        factor_func: 因子计算函数，接受参数返回因子 DataFrame
        threshold: 多空阈值（默认 0）
        """
        results = []
        
        for param_val in param_range:
            # 计算因子
            factor = factor_func(param_val, **kwargs)
            
            # 生成信号
            signals = self.generate_signals(factor, long_threshold=threshold)
            
            # 计算策略收益
            self.calculate_strategy_returns(signals)
            
            # 计算绩效
            perf = self.calculate_performance()
            
            results.append({
                param_name: param_val,
                'sharpe': perf['sharpe_ratio'],
                'annual_return': perf['annual_return'],
                'max_drawdown': perf['max_drawdown'],
                'win_rate': perf['win_rate']
            })
        
        return pd.DataFrame(results)
    # **kwargs 表示可以接受额外参数，如果存在的话
    
    # ============================================================
    # 使用 EWMA 的因子函数示例
    # ============================================================
    
    def momentum_factor_ewma(span: int, **kwargs) -> pd.DataFrame:
        """EWMA 动量因子：价格 / EWMA 均价 - 1"""
        prices = kwargs.get('prices')  # 需要传入 prices
        ema = prices.ewm(span=span, adjust=False).mean()
        return prices / ema - 1
    
    # 使用示例：
    # bt = FactorBacktest(prices)
    # results = bt.parameter_sensitivity(
    #     'span', range(5, 125, 5),
    #     lambda s: bt.prices / bt.prices.ewm(span=s, adjust=False).mean() - 1
    # )